# Naive Bayes Classifier from Scratch
by AI@UCI

** ENSURE YOU ARE RUNNING THIS IN AN ENVIRONMENT WITH THE REQUIRED PACKAGES **

## Introduction to Naive Bayes Classifier

The Naive Bayes Classifier is a probabilistic classification algorithm that:

- Uses Bayes' theorem with strong independence assumptions between features
- Calculates the probability of each class given the features
- Is fast, simple, and works well with small datasets
- Often performs surprisingly well despite the "naive" assumption

We'll implement:
- Gaussian Naive Bayes for continuous features
- Probability calculations using Bayes' theorem
- Classification based on maximum posterior probability

**Let's start by understanding what we're building!**


In [ ]:
! pip install numpy pandas matplotlib


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt, pi, exp

print("✅ All packages imported successfully!")
print("We'll use these libraries to:")
print("- numpy: for numerical computations and array operations")
print("- pandas: for data loading and manipulation")
print("- matplotlib: for creating visualizations")
print("- math: for mathematical functions like sqrt, pi, and exp")


## Loading the Weather Dataset

**Question for you:** What do you think the weather dataset might contain? 
Think about what features would be useful for predicting whether someone should play outside!

Let's load the data and explore it together.


In [ ]:
# Load the weather dataset
WEATHER_CSV = "../datasets/bayes_weather_dataset.csv"
df = pd.read_csv(WEATHER_CSV)
print("📊 Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print("\nFirst few rows:")
df.head()


In [ ]:
# Convert categorical features to numeric
def encode_categorical(df):
    """Convert categorical columns to numeric."""
    df_encoded = df.copy()
    for col in df_encoded.columns:
        if df_encoded[col].dtype == 'object':
            df_encoded[col] = pd.Categorical(df_encoded[col]).codes
    return df_encoded

df_encoded = encode_categorical(df)
print("🔄 Categorical features encoded to numbers!")
print("Encoded dataset:")
print(df_encoded.head())
print(f"\nDataset shape: {df_encoded.shape}")
print(f"Target classes: {df_encoded['play'].unique()}")
print("\n💡 Notice how categorical values like 'sunny', 'overcast', 'rainy' became numbers 0, 1, 2")


In [ ]:
# Separate features and target
X = df_encoded.drop(columns=['play']).values
y = df_encoded['play'].values

print("🎯 Features and target separated!")
print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("Feature names:", df_encoded.drop(columns=['play']).columns.tolist())
print("\n🤔 Can you guess what each feature represents?")
print("Hint: We have 4 features and we're trying to predict if someone should 'play' outside")


## Naive Bayes Implementation

**Time to build our classifier!** 

The Naive Bayes algorithm works by:
1. **Learning**: Calculate probabilities from training data
2. **Predicting**: Use Bayes' theorem to classify new data

**Key concept**: We assume features are independent (that's the "naive" part!)
This means: P(outlook, temperature, humidity, windy | play) = P(outlook | play) × P(temperature | play) × P(humidity | play) × P(windy | play)

Let's implement this step by step!


In [ ]:
class GaussianNaiveBayes:
    def __init__(self):
        """Initialize Gaussian Naive Bayes Classifier."""
        self.classes = None
        self.class_priors = None
        self.means = None
        self.variances = None
        print("🏗️  GaussianNaiveBayes class initialized!")
        print("This will store our learned probabilities and parameters.")

    def fit(self, X, y):
        """Fit the Naive Bayes model."""
        print("📚 Learning from training data...")
        self.classes = np.unique(y)
        n_classes = len(self.classes)
        n_features = X.shape[1]
        
        print(f"Found {n_classes} classes: {self.classes}")
        print(f"Number of features: {n_features}")
        
        # Initialize arrays
        self.class_priors = np.zeros(n_classes)
        self.means = np.zeros((n_classes, n_features))
        self.variances = np.zeros((n_classes, n_features))
        
        # Calculate priors, means, and variances for each class
        for i, cls in enumerate(self.classes):
            mask = y == cls
            X_cls = X[mask]
            
            # Prior probability
            self.class_priors[i] = len(X_cls) / len(X)
            
            # Mean and variance for each feature
            self.means[i] = np.mean(X_cls, axis=0)
            self.variances[i] = np.var(X_cls, axis=0)
            
            print(f"Class {cls}: {len(X_cls)} samples, prior = {self.class_priors[i]:.3f}")
        
        print("✅ Learning complete! Model parameters calculated.")

    def _gaussian_pdf(self, x, mean, var):
        """Calculate Gaussian probability density function."""
        if var == 0:
            return 1.0 if x == mean else 0.0
        return (1 / sqrt(2 * pi * var)) * exp(-((x - mean) ** 2) / (2 * var))

    def _predict_one(self, x):
        """Predict class for a single sample."""
        posteriors = []
        
        for i, cls in enumerate(self.classes):
            # Start with prior probability
            posterior = self.class_priors[i]
            
            # Multiply by likelihood for each feature
            for j in range(len(x)):
                likelihood = self._gaussian_pdf(x[j], self.means[i][j], self.variances[i][j])
                posterior *= likelihood
            
            posteriors.append(posterior)
        
        # Return class with highest posterior probability
        return self.classes[np.argmax(posteriors)]

    def predict(self, X):
        """Predict classes for multiple samples."""
        return np.array([self._predict_one(x) for x in X])

    def score(self, X, y):
        """Calculate accuracy score."""
        preds = self.predict(X)
        return float(np.mean(preds == y))

print("🎉 GaussianNaiveBayes class defined!")
print("This class implements the complete Naive Bayes algorithm.")


In [ ]:
# Train-test split
print("🔄 Splitting data into training and test sets...")
np.random.seed(42)
indices = np.random.permutation(len(X))
train_size = int(0.7 * len(X))
train_idx, test_idx = indices[:train_size], indices[train_size:]

X_train, y_train = X[train_idx], y[train_idx]
X_test, y_test = X[test_idx], y[test_idx]

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print("💡 We use 70% for training and 30% for testing our model")


In [ ]:
# Train and test the Naive Bayes Classifier
print("🚀 Training our Naive Bayes classifier...")
nb_clf = GaussianNaiveBayes()
nb_clf.fit(X_train, y_train)

print("\n🧪 Testing our classifier...")
accuracy = nb_clf.score(X_test, y_test)
print(f"🎯 Test accuracy: {accuracy:.3f} ({accuracy*100:.1f}%)")

# Display model parameters
print("\n📊 Model parameters learned:")
print("Class priors:", nb_clf.class_priors)
print("Means:\n", nb_clf.means)
print("Variances:\n", nb_clf.variances)
print("\n💭 The model learned the probability distributions for each feature in each class!")


## Summary

**🎉 Congratulations! You've successfully implemented and trained a Naive Bayes Classifier!**

### What we accomplished:
1. **Loaded and preprocessed** the weather dataset
2. **Implemented** the complete Gaussian Naive Bayes algorithm from scratch
3. **Trained** the model on weather data
4. **Tested** the model and achieved good accuracy

### Key takeaways:
- **Naive Bayes is probabilistic** - it calculates class probabilities using Bayes' theorem
- **"Naive" assumption** - features are assumed to be independent (often not true in practice)
- **Gaussian assumption** - continuous features are assumed to follow normal distributions
- **Fast and simple** - no complex optimization, just probability calculations
- **Works well with small datasets** - often performs surprisingly well despite assumptions

The implementation demonstrates how probabilistic classification works and shows the power of the Naive Bayes approach even with its simplifying assumptions.

**🤔 Think about it:** How might this classifier help in real-world applications? What other datasets could benefit from Naive Bayes?
